In [445]:
import sympy as sy

In [446]:
import itertools

In [447]:
import numpy as np

Grau dos polinômios d e g

In [448]:
dn = 2

In [449]:
gn = 1

In [450]:
num_d = [1, 2, 3]
num_g = [1, 2]

In [451]:
x = sy.symbols("x")

In [452]:
di = sy.symbols(" ".join(f"d{i}"for i in range(dn+1)))
di

(d0, d1, d2)

In [453]:
gi = sy.symbols(" ".join(f"g{i}"for i in range(gn+1)))
gi

(g0, g1)

In [454]:
dx = sum([i*x**e for e, i in enumerate(di)])
dx

d0 + d1*x + d2*x**2

In [455]:
gx = sum([i*x**e for e, i in enumerate(gi)])
gx

g0 + g1*x

In [456]:
sx = gx*dx
sx

(g0 + g1*x)*(d0 + d1*x + d2*x**2)

In [457]:
mk = [x, x - 1, x + 1]

In [458]:
gk_ = [sy.div(gx, q, domain ='QQ')[1] for q in mk]
gk = gk_ + [gx.args[-1].args[0]]
gk

[g0, g0 + g1, g0 - g1, g1]

In [459]:
dk_ = [sy.div(dx, q, domain ='QQ')[1] for q in mk]
dk = dk_ + [dx.args[-1].args[0]]
dk

[d0, d0 + d1 + d2, d0 - d1 + d2, d2]

In [460]:
la = [[d.coeff(c, 1) for c in di] for d in dk]
la

[[1, 0, 0], [1, 1, 1], [1, -1, 1], [0, 0, 1]]

In [461]:
ma = sy.Matrix(la)
ma

Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[0,  0, 1]])

In [462]:
mmk_ = [sy.expand(d[0]*d[1]) for d in itertools.combinations(reversed(mk), len(mk)-1)]
mmk = mmk_ + [sy.expand(np.prod(mk))]
mmk

[x**2 - 1, x**2 + x, x**2 - x, x**3 - x]

Pegando quociente e resto, agora tem q colocar no formato nm+NM=1

In [463]:
mx_div = [sy.div(dv, ds, domain ='QQ') for dv, ds in zip(mmk, mk)]
mx_div

[(x, -1), (x + 2, 2), (x - 2, 2)]

Multiplicar o resto pela matriz G depois
o sinal negativo vai pra matriz G e não pra C

In [464]:
nnk = [1/z[1] for z in mx_div]
nnk

[-1, 1/2, 1/2]

In [465]:
nk = [q[0]*r*(-1) for q, r in zip(mx_div, nnk)]
nk

[x, -x/2 - 1, 1 - x/2]

colocar em forma matricial

In [466]:
lc = [[d.coeff(x, c) for d in mmk] for c in range(len(np.prod(mk).as_poly().all_coeffs()))]
lc

[[-1, 0, 0, 0], [0, 1, -1, -1], [1, 1, 1, 0], [0, 0, 0, 1]]

In [467]:
cc = sy.Matrix(lc)
cc

Matrix([
[-1, 0,  0,  0],
[ 0, 1, -1, -1],
[ 1, 1,  1,  0],
[ 0, 0,  0,  1]])

In [468]:
lbg = [g*r for g, r in zip(gk, nnk+[1])]

In [469]:
bbg = sy.diag(*lbg)
bbg

Matrix([
[-g0,           0,           0,  0],
[  0, g0/2 + g1/2,           0,  0],
[  0,           0, g0/2 - g1/2,  0],
[  0,           0,           0, g1]])

In [470]:
s = sy.MatMul(cc, bbg, ma, sy.Matrix(di))
s

Matrix([
[-1, 0,  0,  0],
[ 0, 1, -1, -1],
[ 1, 1,  1,  0],
[ 0, 0,  0,  1]])*Matrix([
[-g0,           0,           0,  0],
[  0, g0/2 + g1/2,           0,  0],
[  0,           0, g0/2 - g1/2,  0],
[  0,           0,           0, g1]])*Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[0,  0, 1]])*Matrix([
[d0],
[d1],
[d2]])

In [471]:
np.convolve(num_d, num_g)

array([1, 4, 7, 6])

In [472]:
subs = {k: v for k, v in zip(di+gi, num_d + num_g)}
subs

{d0: 1, d1: 2, d2: 3, g0: 1, g1: 2}

In [473]:
si = s.subs(subs)
si

Matrix([
[-1, 0,  0,  0],
[ 0, 1, -1, -1],
[ 1, 1,  1,  0],
[ 0, 0,  0,  1]])*Matrix([
[-1,   0,    0, 0],
[ 0, 3/2,    0, 0],
[ 0,   0, -1/2, 0],
[ 0,   0,    0, 2]])*Matrix([
[1,  0, 0],
[1,  1, 1],
[1, -1, 1],
[0,  0, 1]])*Matrix([
[1],
[2],
[3]])

In [474]:
sy.expand(sx)

d0*g0 + d0*g1*x + d1*g0*x + d1*g1*x**2 + d2*g0*x**2 + d2*g1*x**3

In [475]:
se = sy.MatMul(cc, bbg, ma, sy.Matrix(di), evaluate=True)
se

Matrix([
[        d0*g0],
[d0*g1 + d1*g0],
[d1*g1 + d2*g0],
[        d2*g1]])

In [476]:
se.subs(subs)

Matrix([
[1],
[4],
[7],
[6]])